In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [33]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [23]:
import bayesflow as bf

In [36]:
from src.simulations.molecules import MoleculeSimulator
from src.simulations.benchmarks.ethene import ethene, ethene_configs
from pathlib import Path

In [37]:
checkpoint_path = Path("checkpoints")
checkpoint_path.mkdir(parents=True, exist_ok=True)

In [25]:
ethene_simulator = MoleculeSimulator(
    molecule_fun=ethene,
    basis="cc-pVDZ",
    coord_scale=1.,
    cache_integrals=False
)

In [26]:
# Prior function to sample geometric configuration for ethene
ethene_configs = ethene_configs

In [158]:
samples = ethene_simulator.sample(
    num_samples=2,
    molecule_config=ethene_configs,
    molecule_kwargs={ "perturb": False },
    include_kwargs={
        "include_hartree_fock": True,
        "include_configs": True,
    }
)

Generating samples: 100%|██████████| 2/2 [00:01<00:00,  1.13it/s]


In [159]:
print(samples.keys())

dict_keys(['occupancies', 'determinant', 'hf_energy', 'cc_bond_distance', 'ch_bond_distance', 'hch_angle', 'twist_angle'])


In [160]:
for k, v in samples.items():
    print(f"{k}: {v.shape}")

occupancies: (2, 48)
determinant: (2, 48, 48)
hf_energy: (2,)
cc_bond_distance: (2,)
ch_bond_distance: (2,)
hch_angle: (2,)
twist_angle: (2,)


In [165]:
inference_variables = ["cc_bond_distance", "ch_bond_distance", "hch_angle", "twist_angle", "hf_energy"]
inference_conditions = ["determinant", "occupancies"]

adapter = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .expand_dims("occupancies", axis=-1)
    .concatenate(inference_conditions, into="inference_conditions")
    .expand_dims(inference_variables, axis=1)
    .concatenate(inference_variables, into="inference_variables")
    .expand_dims("inference_variables", axis=-1)
)

In [166]:
adapted_sims = adapter(ethene_simulator.sample(
    num_samples=2,
    molecule_config=ethene_configs,
    molecule_kwargs={ "perturb": False },
    include_kwargs={
        "include_hartree_fock": True,
        "include_configs": True,
    }
))

Generating samples: 100%|██████████| 2/2 [00:01<00:00,  1.22it/s]


In [167]:
for k, v in adapted_sims.items():
    print(k, v.shape)

inference_conditions (2, 48, 49)
inference_variables (2, 5, 1)


In [168]:
flow_matching = bf.networks.FlowMatching()

In [169]:
flow_matching_workflow = bf.workflows.BasicWorkflow(
    simulator=ethene_simulator,
    adapter=adapter,
    inference_network=flow_matching,
    checkpoint_filepath=checkpoint_path / "ethene_flow_matching.ckpt"
)

In [170]:
training_set = ethene_simulator.sample(
    num_samples=1000,
    molecule_config=ethene_configs,
    molecule_kwargs={ "perturb": False },
    include_kwargs={
        "include_hartree_fock": True,
        "include_configs": True,
    })

Generating samples: 100%|██████████| 1000/1000 [13:14<00:00,  1.26it/s]


In [151]:
validation_set = ethene_simulator.sample(
    num_samples=100,
    molecule_config=ethene_configs,
    molecule_kwargs={ "perturb": False },
    include_kwargs={
        "include_hartree_fock": True,
        "include_configs": True,
    }
)

Generating samples: 100%|██████████| 100/100 [01:22<00:00,  1.21it/s]


In [156]:
print(validation_set.keys())

dict_keys(['nuc_attraction', 'overlaps', 'occupancies', 'determinant', 'hf_energy', 'cc_bond_distance', 'ch_bond_distance', 'hch_angle', 'twist_angle'])


In [157]:
history = flow_matching_workflow.fit_offline(
    data=training_set,
    val_data=validation_set,
    epochs=100,
    batch_size=32
)

INFO:bayesflow:Fitting on dataset instance of OfflineDataset.


TypeError: 'NoneType' object is not callable